# CRRT Baseline Model Data Preparation: 1-Day, 2-Day, 7-Day Frameworks

Computes **per-feature statistical summaries** (no temporal data) from both
**noised** and **denoised** pressure signals, for three prediction frameworks.
Outputs are flat feature vectors saved as CSV files ready for any baseline model (LR, RF, XGB, etc.).

## Statistics computed per feature per day
mean, std, min, max, median, 25th pct, 75th pct, IQR, range, skewness, kurtosis, CV (std/mean), n_valid

## Input per framework
| Framework | Input window | Label |
|---|---|---|
| 1-Day rolling | Stats from **1 day** | `label_1day` |
| 2-Day rolling | Stats from **1 day** | `label_2day` |
| 7-Day non-rolling | Stats from **Days 1–5** concatenated per day | `label_7day` |

For 7-Day: each day (1–5) is summarized separately → feature vector = 5 × 13 stats × 4 features = 260 features.

## Split
70 : 15 : 15 at **PID level**, stratified by `Mortality`. Same split applied to all frameworks and both variants.

## Output
```
Data/Processed/Model_Ready/
  splits/pid_splits.json
  {1day|2day|7day}/
    {noised|denoised}/
      train.csv  |  val.csv  |  test.csv
```
Each CSV row = one sample with all stat features + `label`, `PID`, `Visit_no`, `Day` (or `Visit_no` for 7-day), `Mortality`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as scipy_stats
from sklearn.model_selection import train_test_split

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DATA = Path("../../Data")   # repository Data/ folder (see docs/DATA_PROCESSING.md)

PRESSURE_DENOISED = BASE_DATA / "Processed" / "Machine" / "pressure" / "Denoised" / "CRRT_pressure_wo_peak.csv"
PRESSURE_NOISED   = BASE_DATA / "Processed" / "Machine" / "pressure" / "Noised"   / "CRRT_pressure_with_peak.csv"
LABEL_1DAY        = BASE_DATA / "Processed" / "Labels" / "labels_1day.csv"
LABEL_2DAY        = BASE_DATA / "Processed" / "Labels" / "labels_2day.csv"
LABEL_7DAY        = BASE_DATA / "Processed" / "Labels" / "labels_7day.csv"
OUTPUT_ROOT       = BASE_DATA / "Processed" / "Model_Ready"

FEAT_COLS    = ['ACCESS_PRESSURE', 'FILTER_PRESSURE', 'EFFLUENT_PRESSURE', 'RETURN_PRESSURE']
TRAIN_RATIO  = 0.70
RANDOM_STATE = 42

print("Paths configured.")

## Load Data

In [ ]:
labels_1day = pd.read_csv(LABEL_1DAY)
labels_2day = pd.read_csv(LABEL_2DAY)
labels_7day = pd.read_csv(LABEL_7DAY)

print("Loading pressure data...")
pressure_denoised = pd.read_csv(PRESSURE_DENOISED)
pressure_noised   = pd.read_csv(PRESSURE_NOISED)

print(f"Denoised pressure : {pressure_denoised.shape}")
print(f"Noised pressure   : {pressure_noised.shape}")
print(f"Labels 1-day      : {labels_1day.shape}  | label=1: {(labels_1day['label_1day']==1).sum():,}")
print(f"Labels 2-day      : {labels_2day.shape}  | label=1: {(labels_2day['label_2day']==1).sum():,}")
print(f"Labels 7-day      : {labels_7day.shape}  | label=1: {(labels_7day['label_7day']==1).sum():,}")

## PID-Level Stratified Split (70 : 15 : 15)

In [ ]:
pid_mortality = labels_1day[['PID', 'Mortality']].drop_duplicates('PID').set_index('PID')['Mortality']

all_pids = pid_mortality.index.to_numpy()
strat    = pid_mortality.values

train_pids, temp_pids, _, temp_strat = train_test_split(
    all_pids, strat, test_size=(1 - TRAIN_RATIO),
    stratify=strat, random_state=RANDOM_STATE
)
val_pids, test_pids = train_test_split(
    temp_pids, test_size=0.5,
    stratify=temp_strat, random_state=RANDOM_STATE
)

pid_split = {pid: 'train' for pid in train_pids}
pid_split.update({pid: 'val'  for pid in val_pids})
pid_split.update({pid: 'test' for pid in test_pids})

print(f"Train PIDs: {len(train_pids):,}  |  Val: {len(val_pids):,}  |  Test: {len(test_pids):,}")
for split_name, pids in [('train', train_pids), ('val', val_pids), ('test', test_pids)]:
    mort = pid_mortality.loc[pids]
    print(f"  {split_name:5s}: mort=1 → {mort.sum():3d}/{len(pids):3d} ({100*mort.mean():.1f}%)")

split_dir = OUTPUT_ROOT / 'splits'
split_dir.mkdir(parents=True, exist_ok=True)
with open(split_dir / 'pid_splits.json', 'w') as f:
    json.dump({'train': train_pids.tolist(), 'val': val_pids.tolist(), 'test': test_pids.tolist()}, f, indent=2)
print(f"\nSplit saved to: {split_dir / 'pid_splits.json'}")

## Statistical Feature Computation

For each feature column in one day/window of data, computes 13 statistics:
`mean, std, min, max, median, q25, q75, iqr, range, skewness, kurtosis, cv, n_valid`

Result is a flat dict with keys like `ACCESS_PRESSURE_mean`, `ACCESS_PRESSURE_std`, etc.

In [ ]:
STAT_NAMES = ['mean', 'std', 'min', 'max', 'median', 'q25', 'q75', 'iqr', 'range',
              'skewness', 'kurtosis', 'cv', 'n_valid']


def compute_stats(values: np.ndarray) -> dict:
    """Compute 13 statistics for a 1-D array of valid (non-NaN) values."""
    v = values[~np.isnan(values)]
    n = len(v)
    if n == 0:
        return {s: np.nan for s in STAT_NAMES}

    mean_v = float(np.mean(v))
    std_v  = float(np.std(v, ddof=1)) if n > 1 else 0.0
    q25, median_v, q75 = np.percentile(v, [25, 50, 75])

    return {
        'mean'    : mean_v,
        'std'     : std_v,
        'min'     : float(np.min(v)),
        'max'     : float(np.max(v)),
        'median'  : float(median_v),
        'q25'     : float(q25),
        'q75'     : float(q75),
        'iqr'     : float(q75 - q25),
        'range'   : float(np.max(v) - np.min(v)),
        'skewness': float(scipy_stats.skew(v))      if n > 2 else np.nan,
        'kurtosis': float(scipy_stats.kurtosis(v))  if n > 3 else np.nan,
        'cv'      : float(std_v / mean_v)           if mean_v != 0 else np.nan,
        'n_valid' : int(n),
    }


def day_feature_vector(day_rows: pd.DataFrame, prefix: str = '') -> dict:
    """
    Compute stats for all 4 pressure features from one day's rows.
    Returns a flat dict with keys like 'ACCESS_PRESSURE_mean' (or '{prefix}_ACCESS_PRESSURE_mean').
    """
    result = {}
    for feat in FEAT_COLS:
        vals = day_rows[feat].to_numpy(dtype=float)
        stat_dict = compute_stats(vals)
        col_prefix = f"{prefix}_{feat}" if prefix else feat
        for stat_name, stat_val in stat_dict.items():
            result[f"{col_prefix}_{stat_name}"] = stat_val
    return result


print(f"Stats per feature : {len(STAT_NAMES)}")
print(f"Features          : {len(FEAT_COLS)}")
print(f"Total cols (1-day): {len(STAT_NAMES) * len(FEAT_COLS)}")
print(f"Total cols (7-day): {len(STAT_NAMES) * len(FEAT_COLS) * 5}  (5 days concatenated)")

In [ ]:
# Build fast lookup dict: (PID, Visit_no, Day) -> subset DataFrame
print("Building lookup index...")

def build_lookup(pressure_df):
    return {
        (int(pid), int(visit), int(day)): grp
        for (pid, visit, day), grp in pressure_df.groupby(['PID', 'Visit_no', 'Day'])
    }

lookup_denoised = build_lookup(pressure_denoised)
lookup_noised   = build_lookup(pressure_noised)
print(f"Denoised lookup: {len(lookup_denoised):,} (PID, Visit, Day) entries")
print(f"Noised lookup  : {len(lookup_noised):,} (PID, Visit, Day) entries")

## 1-Day and 2-Day Rolling: Build Feature Tables

In [ ]:
def build_rolling_features(label_df, label_col, lookup):
    """
    For each labeled (PID, Visit_no, Day), compute statistical features
    from that single day's pressure data.

    Returns a DataFrame with one row per sample:
      meta columns + 52 stat feature columns + label column.
    """
    records = []
    skipped = 0

    for _, row in label_df.iterrows():
        pid   = int(row['PID'])
        visit = int(row['Visit_no'])
        day   = int(row['Day'])
        label = int(row[label_col])
        mort  = int(row['Mortality'])
        dud   = row.get('days_until_death', np.nan)

        key = (pid, visit, day)
        if key not in lookup:
            skipped += 1
            continue

        feat_dict = day_feature_vector(lookup[key])
        feat_dict.update({
            'PID'             : pid,
            'Visit_no'        : visit,
            'Day'             : day,
            'Mortality'       : mort,
            'days_until_death': dud,
            'label'           : label,
        })
        records.append(feat_dict)

    print(f"  Built {len(records):,} samples | Skipped {skipped:,}")
    meta_cols = ['PID', 'Visit_no', 'Day', 'Mortality', 'days_until_death', 'label']
    feat_cols_out = [c for c in records[0] if c not in meta_cols]
    return pd.DataFrame(records)[meta_cols + feat_cols_out]


print("build_rolling_features defined.")

In [ ]:
def save_split_csvs(df, framework_name, variant_name):
    """Attach split column and save train/val/test CSVs."""
    df = df.copy()
    df['split'] = df['PID'].map(pid_split)

    # Drop rows whose PID has no split (PIDs not in label_1day)
    df = df[df['split'].notna()]

    out_dir = OUTPUT_ROOT / framework_name / variant_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for split_name in ['train', 'val', 'test']:
        subset = df[df['split'] == split_name].drop(columns='split').reset_index(drop=True)
        out_path = out_dir / f"{split_name}.csv"
        subset.to_csv(out_path, index=False)
        n0 = (subset['label'] == 0).sum()
        n1 = (subset['label'] == 1).sum()
        print(f"  {split_name:5s}: {len(subset):,} rows | label=0: {n0:,}  label=1: {n1:,} → {out_path.name}")


print("save_split_csvs defined.")

In [ ]:
# ── 1-Day framework ────────────────────────────────────────────────────────
print("=" * 60)
print("1-DAY ROLLING — DENOISED")
print("=" * 60)
df_1day_denoised = build_rolling_features(labels_1day, 'label_1day', lookup_denoised)
save_split_csvs(df_1day_denoised, '1day', 'denoised')

print()
print("=" * 60)
print("1-DAY ROLLING — NOISED")
print("=" * 60)
df_1day_noised = build_rolling_features(labels_1day, 'label_1day', lookup_noised)
save_split_csvs(df_1day_noised, '1day', 'noised')

In [ ]:
# ── 2-Day framework ────────────────────────────────────────────────────────
print("=" * 60)
print("2-DAY ROLLING — DENOISED")
print("=" * 60)
df_2day_denoised = build_rolling_features(labels_2day, 'label_2day', lookup_denoised)
save_split_csvs(df_2day_denoised, '2day', 'denoised')

print()
print("=" * 60)
print("2-DAY ROLLING — NOISED")
print("=" * 60)
df_2day_noised = build_rolling_features(labels_2day, 'label_2day', lookup_noised)
save_split_csvs(df_2day_noised, '2day', 'noised')

## 7-Day Non-Rolling: Build Feature Tables

For each `(PID, Visit_no)`, stats are computed **per day** for Days 1–5 separately,
then concatenated into one feature vector.
Column naming: `day1_ACCESS_PRESSURE_mean`, `day2_ACCESS_PRESSURE_mean`, … etc.

Total feature columns = 5 days × 4 features × 13 stats = **260 features**.

In [ ]:
def build_7day_features(label_df, label_col, lookup):
    """
    For each (PID, Visit_no), compute per-day stats for Days 1-5
    and concatenate into one row.

    Returns a DataFrame with one row per visit.
    """
    # One row per visit — label is shared across all days of the visit
    visit_labels = (
        label_df.groupby(['PID', 'Visit_no'])
        .agg(label=(label_col, 'first'), Mortality=('Mortality', 'first'))
        .reset_index()
    )

    records = []
    skipped = 0

    for _, vrow in visit_labels.iterrows():
        pid   = int(vrow['PID'])
        visit = int(vrow['Visit_no'])
        label = int(vrow['label'])
        mort  = int(vrow['Mortality'])

        feat_dict = {}
        all_present = True

        for d in range(1, 6):
            key = (pid, visit, d)
            if key not in lookup:
                all_present = False
                break
            day_stats = day_feature_vector(lookup[key], prefix=f'day{d}')
            feat_dict.update(day_stats)

        if not all_present:
            skipped += 1
            continue

        feat_dict.update({
            'PID'      : pid,
            'Visit_no' : visit,
            'Mortality': mort,
            'label'    : label,
        })
        records.append(feat_dict)

    print(f"  Built {len(records):,} visit samples | Skipped {skipped:,}")
    meta_cols = ['PID', 'Visit_no', 'Mortality', 'label']
    feat_cols_out = [c for c in records[0] if c not in meta_cols]
    return pd.DataFrame(records)[meta_cols + feat_cols_out]


print("build_7day_features defined.")

In [ ]:
# ── 7-Day framework ────────────────────────────────────────────────────────
print("=" * 60)
print("7-DAY NON-ROLLING — DENOISED")
print("=" * 60)
df_7day_denoised = build_7day_features(labels_7day, 'label_7day', lookup_denoised)
save_split_csvs(df_7day_denoised, '7day', 'denoised')

print()
print("=" * 60)
print("7-DAY NON-ROLLING — NOISED")
print("=" * 60)
df_7day_noised = build_7day_features(labels_7day, 'label_7day', lookup_noised)
save_split_csvs(df_7day_noised, '7day', 'noised')

## Summary

In [ ]:
print("=" * 72)
print("FINAL SUMMARY")
print("=" * 72)
print(f"{'Framework':<8} {'Variant':<10} {'Split':<7} {'Samples':>9} {'label=0':>9} {'label=1':>9}")
print("-" * 72)

for fw in ['1day', '2day', '7day']:
    for variant in ['denoised', 'noised']:
        for split in ['train', 'val', 'test']:
            p = OUTPUT_ROOT / fw / variant / f"{split}.csv"
            if p.exists():
                df_tmp = pd.read_csv(p, usecols=['label'])
                n0 = (df_tmp['label'] == 0).sum()
                n1 = (df_tmp['label'] == 1).sum()
                print(f"{fw:<8} {variant:<10} {split:<7} {len(df_tmp):>9,} {n0:>9,} {n1:>9,}")
            else:
                print(f"{fw:<8} {variant:<10} {split:<7} {'MISSING':>9}")

print("=" * 72)
print(f"\nFeature counts:")
print(f"  1-day / 2-day : {len(STAT_NAMES) * len(FEAT_COLS)} features  ({len(FEAT_COLS)} feats x {len(STAT_NAMES)} stats)")
print(f"  7-day         : {len(STAT_NAMES) * len(FEAT_COLS) * 5} features  (5 days x {len(FEAT_COLS)} feats x {len(STAT_NAMES)} stats)")

In [ ]:
# Spot-check: show first row of one file to confirm column structure
sample_path = OUTPUT_ROOT / '1day' / 'denoised' / 'train.csv'
if sample_path.exists():
    sample = pd.read_csv(sample_path, nrows=1)
    print(f"Columns ({len(sample.columns)}):")
    for col in sample.columns:
        print(f"  {col:45s}  {sample[col].iloc[0]}")